Chuẩn bị

In [16]:
import os
import numpy as np

from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

Chuẩn bị tài liệu

In [17]:
corpus = [
    {
        "source_id": "IT-001",
        "text": """
Nhân viên phải thay đổi mật khẩu tài khoản công ty ít nhất mỗi 90 ngày.
Mật khẩu phải có tối thiểu 12 ký tự, bao gồm chữ hoa, chữ thường, số và ký tự đặc biệt.
Không được sử dụng lại ba mật khẩu gần nhất.
"""
    },

    {
        "source_id": "IT-002",
        "text": """
Không được cắm USB cá nhân vào máy tính công ty.
Nếu cần sao chép dữ liệu phải sử dụng USB do phòng CNTT cấp.
"""
    },

    {
        "source_id": "IT-003",
        "text": """
Khi làm việc từ xa, nhân viên bắt buộc kết nối VPN của công ty trước khi truy cập hệ thống nội bộ.
"""
    },

    {
        "source_id": "IT-004",
        "text": """
Email chứa thông tin mật không được gửi cho địa chỉ email bên ngoài công ty nếu chưa được mã hóa.
"""
    },

    {
        "source_id": "IT-005",
        "text": """
Máy tính xách tay của công ty phải được khóa màn hình khi người dùng rời khỏi chỗ ngồi.
"""
    },

    {
        "source_id": "IT-006",
        "text": """
Dữ liệu quan trọng phải được sao lưu hàng ngày lên hệ thống lưu trữ của công ty.
Không lưu dữ liệu quan trọng trên thiết bị cá nhân.
"""
    },

    {
        "source_id": "IT-007",
        "text": """
Khi phát hiện sự cố an toàn thông tin, nhân viên phải báo ngay cho bộ phận CNTT trong vòng 30 phút.
"""
    },

    {
        "source_id": "IT-008",
        "text": """
Phần mềm trên máy tính công ty chỉ được cài đặt sau khi được phòng CNTT phê duyệt.
"""
    }
]

Chia chuncks

In [18]:
chunks = []

for doc in corpus:
    chunks.append({
        "source_id": doc["source_id"],
        "text": doc["text"]
    })

Embedding

In [19]:
def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )

    return response.data[0].embedding

for chunk in chunks:
    chunk["embedding"] = get_embedding(chunk["text"])

Vector Search (Retrieval)

In [20]:
def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)

    return np.dot(vec1, vec2) / (
        np.linalg.norm(vec1) * np.linalg.norm(vec2)
    )

def retrieve(query, k=3):
    query_embedding = get_embedding(query)

    scores = []

    for chunk in chunks:
        score = cosine_similarity(
            query_embedding,
            chunk["embedding"]
        )

        scores.append({
            "score": score,
            "source_id": chunk["source_id"],
            "text": chunk["text"]
        })

    scores.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return scores[:k]

Sinh câu trả lời (Generation)

In [21]:
def generate_answer(query, results):
    context = ""

    for item in results:
        context += (
            f"[{item['source_id']}]\n"
            f"{item['text']}\n\n"
        )

    prompt = f"""
Bạn là trợ lý trả lời câu hỏi dựa trên tài liệu nội bộ.

Chỉ được trả lời dựa trên ngữ cảnh dưới đây.

Nếu ngữ cảnh không chứa câu trả lời thì hãy trả lời đúng:

"Không tìm thấy trong tài liệu."

Luôn trích nguồn dưới dạng [source_id].

Ngữ cảnh:

{context}

Câu hỏi:

{query}
"""

    response = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt
    )

    return response.output_text

Hàm answer(query)

In [22]:
def answer(query, k=3):
    results = retrieve(query, k)

    response = generate_answer(query, results)

    sources = []

    for item in results:
        sources.append(item["source_id"])

    return response, sources

Cell test

In [ ]:
test_questions = [
    "Có được dùng USB cá nhân không?",
    "Bao lâu phải đổi mật khẩu?",
    "Có bắt buộc VPN không?",
    "Có được gửi email chứa thông tin mật ra ngoài không?",
    "Có được cài phần mềm lên máy công ty không?",
    "Khi phát hiện sự cố bảo mật thì phải làm gì?",
    "Có được lưu dữ liệu trên thiết bị cá nhân không?"
]

for i, question in enumerate(test_questions, 1):
    print("=" * 80)
    print(f"Test {i}")
    print(f"Question: {question}")

    response, sources = answer(question)

    print("\nAnswer:")
    print(response)

    print("\nSources:")
    print(sources)
    print()

Test 1
Question: Có được dùng USB cá nhân không?

Answer:
Không được cắm USB cá nhân vào máy tính công ty. Nếu cần sao chép dữ liệu phải sử dụng USB do phòng CNTT cấp [IT-002].

Sources:
['IT-002', 'IT-006', 'IT-008']

Test 2
Question: Bao lâu phải đổi mật khẩu?

Answer:
Nhân viên phải thay đổi mật khẩu tài khoản công ty ít nhất mỗi 90 ngày. [IT-001]

Sources:
['IT-001', 'IT-003', 'IT-007']

Test 3
Question: Có bắt buộc VPN không?

Answer:
Có, khi làm việc từ xa, nhân viên bắt buộc kết nối VPN của công ty trước khi truy cập hệ thống nội bộ [IT-003].

Sources:
['IT-003', 'IT-008', 'IT-002']

Test 4
Question: Có được gửi email chứa thông tin mật ra ngoài không?
